# Credit Card Fraud Detection — EDA & Modeling Walkthrough

This notebook walks through data inspection, exploratory data analysis, preprocessing, imbalance handling, model training, and evaluation. It mirrors the pipeline in `src/` but is meant for interactive exploration and for generating the charts used in the presentation.

**Before running:** place `creditcard.csv` in `../data/` (see `data/README.md` for download instructions).

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import load_data, inspect_data, clean_data, split_data, scale_features, balance_with_smote, get_class_weights
from evaluate import evaluate_model, compare_models

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load and inspect the data

In [ ]:
df = load_data('../data/creditcard.csv')
df.head()

In [ ]:
report = inspect_data(df)
for k, v in report.items():
    print(f"{k}: {v}")

Observation: the dataset is almost entirely legitimate transactions — fraud makes up roughly 0.17% of all rows. This single fact drives most of the modeling decisions later in this notebook.

## 2. Clean the data (duplicates, missing values)

In [ ]:
df = clean_data(df)
print(df.shape)

## 3. Exploratory Data Analysis

### 3.1 Class distribution

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x='Class', data=df)
plt.title('Legitimate (0) vs Fraudulent (1) Transactions')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

print(df['Class'].value_counts(normalize=True) * 100)

**What this tells us:** the bar for fraud (1) is barely visible next to legitimate (0) — this is the class imbalance problem in one picture.

### 3.2 Transaction amount distribution (all transactions)

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df['Amount'], bins=100, kde=False)
plt.title('Distribution of Transaction Amounts (All)')
plt.xlim(0, 2000)
plt.show()

**What this tells us:** most transactions are small-value, with a long right tail of larger, less frequent transactions — motivates scaling `Amount` before feeding it to models like Logistic Regression.

### 3.3 Amount distribution — fraud vs legitimate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.histplot(df[df['Class']==1]['Amount'], bins=50, ax=axes[0], color='red')
axes[0].set_title('Fraudulent Transaction Amounts')
sns.histplot(df[df['Class']==0]['Amount'], bins=50, ax=axes[1], color='green')
axes[1].set_title('Legitimate Transaction Amounts')
axes[1].set_xlim(0, 2000)
plt.tight_layout()
plt.show()

**What this tells us:** fraudulent transactions skew toward lower amounts with occasional outliers, somewhat different from the overall legitimate distribution — a weak but potentially useful signal.

### 3.4 Boxplot of amounts by class

In [ ]:
plt.figure(figsize=(6,5))
sns.boxplot(x='Class', y='Amount', data=df)
plt.ylim(0, 500)
plt.title('Transaction Amount by Class (zoomed)')
plt.show()

**What this tells us:** puts the spread and outliers of both classes side by side, complementing the histograms above.

### 3.5 Correlation heatmap

In [ ]:
plt.figure(figsize=(14,10))
corr = df.corr()
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

print(corr['Class'].sort_values(ascending=False))

**What this tells us:** since V1–V28 are PCA outputs, they're largely decorrelated from each other, but a handful show a noticeably stronger (positive or negative) correlation with `Class` — worth watching in feature-importance results later.

### 3.6 Transaction time distribution

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df['Time'], bins=100)
plt.title('Transaction Time Distribution')
plt.xlabel('Seconds since first transaction')
plt.show()

**What this tells us:** the two humps correspond to the two days of data collection, with dips likely reflecting overnight hours of lower transaction volume — confirms `Time` reflects a real daily cycle.

## 4. Train/test split (before scaling or resampling)

In [ ]:
X_train, X_test, y_train, y_test = split_data(df)
print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)
print('Train fraud rate:', y_train.mean())
print('Test fraud rate:', y_test.mean())

## 5. Scale Time & Amount (fit on train only)

In [ ]:
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)
X_train_scaled[['Time','Amount']].describe()

## 6. Class imbalance handling — two approaches

In [ ]:
weights = get_class_weights(y_train)
print('Class weights:', weights)

X_train_smote, y_train_smote = balance_with_smote(X_train_scaled, y_train)
print('Before SMOTE:', y_train.value_counts().to_dict())
print('After SMOTE :', y_train_smote.value_counts().to_dict())

## 7. Train and compare models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

results = {}

# --- Class-weight versions ---
models_cw = {
    'Logistic Regression (CW)': LogisticRegression(max_iter=1000, class_weight=weights, random_state=42),
    'Decision Tree (CW)': DecisionTreeClassifier(class_weight=weights, max_depth=10, random_state=42),
    'Random Forest (CW)': RandomForestClassifier(n_estimators=200, class_weight=weights, random_state=42, n_jobs=-1),
}

for name, model in models_cw.items():
    model.fit(X_train_scaled, y_train)
    results[name] = evaluate_model(model, X_test_scaled, y_test, model_name=name, save_plots=False)

In [ ]:
# --- SMOTE versions ---
models_smote = {
    'Logistic Regression (SMOTE)': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree (SMOTE)': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest (SMOTE)': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
}

for name, model in models_smote.items():
    model.fit(X_train_smote, y_train_smote)
    results[name] = evaluate_model(model, X_test_scaled, y_test, model_name=name, save_plots=False)

## 8. Compare all models side by side

In [ ]:
compare_models(results)

**Discussion:** compare precision, recall, F1, and ROC-AUC across strategies here based on your actual run's numbers — this is a good table to screenshot for a presentation slide. Generally, watch whether SMOTE improves recall at some cost to precision compared to class weighting.

## 9. Hyperparameter tuning (Random Forest, SMOTE data)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [8, 12, 16, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=8, scoring='f1', cv=3, random_state=42, n_jobs=-1, verbose=1,
)
search.fit(X_train_smote, y_train_smote)
print('Best params:', search.best_params_)

best_rf = search.best_estimator_
final_metrics = evaluate_model(best_rf, X_test_scaled, y_test, model_name='Tuned Random Forest', save_plots=True)

## 10. Save the final model

In [ ]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(best_rf, '../models/fraud_detection_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(list(X_train.columns), '../models/feature_columns.pkl')
print('Saved model, scaler, and feature columns to ../models/')

## 11. Next steps

- Run `python src/predict.py` from the project root for a quick CLI check.
- Run `streamlit run app.py` from the project root for the interactive demo.
- See `PROJECT_REPORT.md` for the full write-up, metric explanations, and viva Q&A.